# Data Cleaning

- I manually selected columns, timeframe for research and countries before cleaning and merged them in an excel file.
Later on, since excel hold some values as dates data was corrupt. 
- Down below I used a code block to convert dates that should be written as numerical values. 

Example 11.11 -> written as 11.November in the dataset.

In [9]:
import pandas as pd

# Excel ay kısaltmaları -> sayı
MONTH_MAP = {
    'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
    'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
    'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'
}

def fix_excel_date(val):
    """11.Nov -> 11.11, 28.Jul -> 28.07 etc."""
    if pd.isna(val):
        return val
    val = str(val)
    for month, num in MONTH_MAP.items():
        if month in val:
            return val.replace(month, num)
    return val

# Load
df = pd.read_csv('real_final_with_correct_countries.csv', sep=';', encoding='utf-8-sig')

# Fix stringencyindex column
df['stringencyindex'] = df['stringencyindex'].apply(fix_excel_date)

# Save
df.to_csv('stringency_12_countries_fixed.csv', sep=';', index=False)

print("Done!")

Done!


In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
# 12 ülke
COUNTRIES = [
    'Italy', 'Germany', 'France', 'United Kingdom', 'Spain', 
    'Turkey', 'Brazil', 'India', 'United States', 'Sweden', 
    'South Korea', 'Japan'
]

# Load
df = pd.read_csv('Global_Mobility_Report.csv')

# Sadece ülke seviyesi (sub_region boş)
df = df[df['sub_region_1'].isna()]

# 12 ülkeyi filtrele
df = df[df['country_region'].isin(COUNTRIES)]

# Sadece gerekli sütunları al
df = df[['country_region_code', 'country_region', 'date',
         'retail_and_recreation_percent_change_from_baseline',
         'grocery_and_pharmacy_percent_change_from_baseline',
         'parks_percent_change_from_baseline',
         'transit_stations_percent_change_from_baseline',
         'workplaces_percent_change_from_baseline',
         'residential_percent_change_from_baseline']]

# Sütun isimlerini değiştir
df.columns = ['country_code', 'country', 'date', 'retail', 'grocery', 'parks', 'transit', 'workplaces', 'residential']

# Save
df.to_csv('mobility_12_countries.csv', index=False)

print(f"Done! {len(df)} rows saved.")

Done! 12662 rows saved.


**FINALLY MERGE DATASETS**

In [14]:
import pandas as pd

# Load mobility (comma separated, YYYY-MM-DD)
mobility = pd.read_csv('mobility_12_countries.csv')
mobility['date'] = pd.to_datetime(mobility['date'])

# Remove duplicates in mobility
mobility = mobility.drop_duplicates(subset=['country', 'date'], keep='first')

# Load stringency (semicolon separated, D.MM.YYYY)
stringency = pd.read_csv('stringency_12_countries_fixed.csv', sep=';')
stringency['date'] = pd.to_datetime(stringency['date'], format='%d.%m.%Y')

# Merge on country name and date
merged = pd.merge(
    mobility, 
    stringency,
    left_on=['country', 'date'],
    right_on=['countryname', 'date'],
    how='inner'
)

# Clean up columns
merged = merged.drop(columns=['countryname', 'countrycode'])
merged = merged.rename(columns={'country_code': 'countrycode'})

# Reorder
cols = ['countrycode', 'country', 'date', 
        'retail', 'grocery', 'parks', 'transit', 'workplaces', 'residential',
        'stringencyindex', 'c1_school_closing', 'c2_workplace_closing', 
        'c3_cancel_public_events', 'c4_restrictions_on_gatherings',
        'c5_close_public_transport', 'c6_stay_at_home_requirements',
        'c7_restrictions_on_internal_movement', 'c8_international_travel_controls',
        'e1_income_support', 'confirmedcases', 'confirmeddeaths']
merged = merged[cols]

# Save
merged.to_csv('covid_merged_final.csv', index=False)

print(f"Done! {len(merged)} rows saved.")

Done! 3872 rows saved.
